In [11]:
#Import Libraries
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kanak\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kanak\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [12]:
#Sample Dataset (25 Reviews)
data = {
    "review": [
        "Amazing product, works perfectly",
        "Very bad experience, waste of money",
        "Excellent quality, highly recommend",
        "Terrible product, broke in one day",
        "Good value for money",
        "Worst purchase ever",
        "Loved it, will buy again",
        "Not worth the price",
        "Superb performance and design",
        "Horrible, totally disappointed",
        "Great quality and fast delivery",
        "Very poor build quality",
        "Satisfied with the product",
        "Do not buy this item",
        "Fantastic experience",
        "Bad quality, not recommended",
        "Really good product",
        "Very disappointing",
        "Best purchase I made",
        "Waste of time and money",
        "Nice and useful",
        "Terrible service",
        "Excellent product",
        "Worst quality",
        "Highly satisfied"
    ],
    "sentiment": [
        1,0,1,0,1,0,1,0,1,0,
        1,0,1,0,1,0,1,0,1,0,
        1,0,1,0,1
    ]
}

df = pd.DataFrame(data)

In [13]:
# Text Preprocessing
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)  # remove punctuation
    words = text.split()
    
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    
    return " ".join(words)

df["clean_review"] = df["review"].apply(preprocess)

In [14]:
#Feature Extraction (TF-IDF)
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),   # bigrams improve accuracy
    max_features=1000
)

X = vectorizer.fit_transform(df["clean_review"])
y = df["sentiment"]

In [15]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
# Train Naïve Bayes Model
model = MultinomialNB(alpha=0.5)  # smoothing improves performance
model.fit(X_train, y_train)

MultinomialNB(alpha=0.5)

In [17]:
# Predictions
y_pred = model.predict(X_test)

In [18]:
#Evaluation
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8

Confusion Matrix:
 [[1 1]
 [0 3]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.75      1.00      0.86         3

    accuracy                           0.80         5
   macro avg       0.88      0.75      0.76         5
weighted avg       0.85      0.80      0.78         5



In [21]:
# Test Custom Input
def predict_sentiment(text):
    text = preprocess(text)
    vector = vectorizer.transform([text])
    prediction = model.predict(vector)[0]
    
    return "Positive" if prediction == 1 else "Negative"

# Example
print("\nCustom Test:", predict_sentiment("This product is really bad, I hate it!"))

# Example
print("\nCustom Test:", predict_sentiment("This product is really amazing"))


Custom Test: Negative

Custom Test: Positive


In [22]:
import pickle

# Save trained model
with open("product_sentiment_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save TF-IDF vectorizer
with open("product_tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!
